In [8]:
from __future__ import annotations
import os
import io
import sys
import pandas as pd
import requests
from typing import Dict, List, Optional, Tuple

from fredapi import Fred
from dash import Dash, dcc, html, Input, Output
import plotly.graph_objs as go
import yfinance as yf

FRED_API_KEY = '3a678ff43b026f921d7198f75e01bd28'
SAVE_DIR = "data_fred"
os.makedirs(SAVE_DIR, exist_ok=True)

START_DATE: Optional[str] = "2008-01-01"
END_DATE:   Optional[str] = None


PREFER_ONLY_FRED: bool = False


SERIES: Dict[str, Dict[str, str]] = {
    # Ставка ФРС (daily)
    "EFFR":      {"name": "Effective Fed Funds Rate",      "unit": "%",        "freq": "D"},

    # Цены/труд/ликвидность (monthly)
    "CPIAUCSL":  {"name": "CPI (All items, SA, 82-84=100)", "unit": "index",   "freq": "M"},
    "UNRATE":    {"name": "Unemployment Rate (SA)",         "unit": "%",       "freq": "M"},
    "M2SL":      {"name": "M2 Money Stock (SA, $bn)",       "unit": "USD bn",  "freq": "M"},

    # Доходности UST (daily)
    "DGS2":      {"name": "UST 2Y Yield",  "unit": "%", "freq": "D"},
    "DGS10":     {"name": "UST 10Y Yield", "unit": "%", "freq": "D"},
    "DGS30":     {"name": "UST 30Y Yield", "unit": "%", "freq": "D"},

    # Сырьё (daily)
    "DCOILBRENTEU": {"name": "Brent Oil (USD/bbl)", "unit": "USD/bbl", "freq": "D"},
    # золото: начально заявляем AM; внутри загрузчика будет перебор AM -> PM -> CSV -> YF
    "GOLDAMGBD228NLBM": {"name": "Gold (London AM Fix, USD/oz)", "unit": "USD/oz", "freq": "D"},

    # Курс USD/EUR (daily)
    "DEXUSEU":   {"name": "USD per EUR (H.10)", "unit": "USD/EUR", "freq": "D"},
}
GOLD_FRED_CODES = ["GOLDAMGBD228NLBM", "GOLDPMGBD228NLBM"]  # порядок попыток


def _to_date(s: Optional[str]) -> Optional[str]:
    if not s:
        return None
    return pd.to_datetime(s).strftime("%Y-%m-%d")

def _fetch_via_fredapi(fred: Fred, code: str) -> pd.DataFrame:
    s = fred.get_series(code, observation_start=START_DATE or None, observation_end=END_DATE or None)
    df = s.to_frame("value").reset_index().rename(columns={"index": "date"})
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df = df.dropna(subset=["value"]).copy()
    return df[["date", "value"]]

def _fetch_via_csv_endpoint(code: str, start: Optional[str], end: Optional[str]) -> pd.DataFrame:
    """
    FRED CSV fallback (не требует ключа).
    https://fred.stlouisfed.org/graph/fredgraph.csv?id=<CODE>&cosd=&coed=
    """
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv"
    params = {"id": code}
    if start: params["cosd"] = _to_date(start)
    if end:   params["coed"] = _to_date(end)
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    if "DATE" not in df.columns or code not in df.columns:
        raise ValueError(f"Unexpected CSV format for {code}")
    df = df.rename(columns={"DATE": "date", code: "value"})
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["value"])
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    return df[["date", "value"]]

def _fetch_gold_via_yahoo(start: Optional[str], end: Optional[str]) -> pd.DataFrame:
    """
    Фоллбэк на spot золото: тикер 'XAUUSD=X' (USD/oz). Берём дневной Close.
    """
    df = yf.download("XAUUSD=X", start=start, end=end, auto_adjust=False, progress=False)
    if df.empty:
        raise RuntimeError("Yahoo returned empty XAUUSD=X")
    out = df[["Close"]].rename(columns={"Close": "value"}).reset_index().rename(columns={"Date": "date"})
    out["date"] = pd.to_datetime(out["date"]).dt.tz_localize(None)
    out = out.dropna(subset=["value"])
    return out[["date", "value"]]

def _gold_name(code: str) -> str:
    if code.endswith("AMGBD228NLBM"):
        return "Gold (London AM Fix, USD/oz)"
    if code.endswith("PMGBD228NLBM"):
        return "Gold (London PM Fix, USD/oz)"
    return "Gold (USD/oz)"

def fetch_series(code: str, fred: Fred) -> Tuple[pd.DataFrame, str]:
    """
    Возвращает (df, source), где source ∈ {FRED_API, FRED_CSV, FRED_LBMA_AM, FRED_LBMA_PM, YF_SPOT}
    df: date, value, code, name, unit, freq, source
    """
    if code not in GOLD_FRED_CODES and code != "GOLDAMGBD228NLBM":
        try:
            df = _fetch_via_fredapi(fred, code)
            src = "FRED_API"
        except Exception as e_api:
            print(f"[WARN] fredapi failed for {code}: {e_api}. Trying FRED CSV …")
            df = _fetch_via_csv_endpoint(code, START_DATE, END_DATE)
            src = "FRED_CSV"

        meta = SERIES.get(code, {"name": code, "unit": "", "freq": ""})
        df["code"] = code
        df["name"] = meta["name"]
        df["unit"] = meta["unit"]
        df["freq"] = meta["freq"]
        df["source"] = src
        return df, src

    for gold_code in GOLD_FRED_CODES:
        try:
            df = _fetch_via_fredapi(fred, gold_code)
            use_code = gold_code
            use_name = _gold_name(gold_code)
            src = "FRED_LBMA_AM" if gold_code.endswith("AMGBD228NLBM") else "FRED_LBMA_PM"
            df["code"] = use_code
            df["name"] = use_name
            df["unit"] = "USD/oz"
            df["freq"] = "D"
            df["source"] = src
            return df, src
        except Exception as e_api:
            print(f"[WARN] fredapi failed for {gold_code}: {e_api}. Trying FRED CSV …")
            try:
                df = _fetch_via_csv_endpoint(gold_code, START_DATE, END_DATE)
                use_code = gold_code
                use_name = _gold_name(gold_code)
                src = "FRED_LBMA_AM" if gold_code.endswith("AMGBD228NLBM") else "FRED_LBMA_PM"
                df["code"] = use_code
                df["name"] = use_name
                df["unit"] = "USD/oz"
                df["freq"] = "D"
                df["source"] = src
                return df, src
            except Exception as e_csv:
                print(f"[WARN] FRED CSV failed for {gold_code}: {e_csv}")

    if PREFER_ONLY_FRED:
        raise RuntimeError("Gold not available from FRED and PREFER_ONLY_FRED=True")

    print("[WARN] FRED gold not available — falling back to Yahoo XAUUSD=X …")
    df = _fetch_gold_via_yahoo(START_DATE, END_DATE)
    use_code = "XAUUSD=X"
    use_name = "Gold Spot (XAUUSD, USD/oz)"
    SERIES[use_code] = {"name": use_name, "unit": "USD/oz", "freq": "D"}
    df["code"] = use_code
    df["name"] = use_name
    df["unit"] = "USD/oz"
    df["freq"] = "D"
    df["source"] = "YF_SPOT"
    return df, "YF_SPOT"

def load_and_save_all() -> Dict[str, pd.DataFrame]:
    fred = Fred(api_key=FRED_API_KEY) if FRED_API_KEY else Fred()
    frames: Dict[str, pd.DataFrame] = {}
    failed: List[str] = []

    print("=== Loading series ===")
    for code in list(SERIES.keys()):
        try:
            df, src = fetch_series(code, fred)
            actual_code = df["code"].iloc[0]
            frames[actual_code] = df[["date", "value", "code", "name", "unit", "freq", "source"]].copy()
            out_path = os.path.join(SAVE_DIR, f"{actual_code}.csv")
            frames[actual_code].to_csv(out_path, index=False)
            print(f"[OK] {actual_code} ({src}) → {out_path} (rows={len(df)})")
        except Exception as e:
            print(f"[ERROR] skipped {code}: {e}")
            failed.append(code)

    if not frames:
        raise SystemExit("No series fetched successfully — check codes/network.")

    all_long = pd.concat(frames.values(), ignore_index=True)
    all_long.sort_values(["code", "date"], inplace=True)
    all_long.to_csv(os.path.join(SAVE_DIR, "all_long.csv"), index=False)

    all_wide = (
        all_long.pivot(index="date", columns="code", values="value")
        .sort_index().reset_index()
    )
    all_wide.to_csv(os.path.join(SAVE_DIR, "all_wide.csv"), index=False)

    if failed:
        print("[WARN] Skipped series:", ", ".join(failed))
    print(f"[DONE] Saved CSVs to: {os.path.abspath(SAVE_DIR)}")
    return frames

TRANSFORMS = {
    "none": "Без трансформации",
    "pct_mom": "Δ% (к предыдущему наблюд.)",
    "pct_yoy": "Δ% YoY (12 мес для monthly; ~252д для daily)",
    "normalize_100": "Нормировка (первое значение = 100)",
}
AGG_OPTIONS = {
    "native": "Нативная частота",
    "M_last": "Месяц: последнее",
    "M_mean": "Месяц: среднее",
}

def pct_change(series: pd.Series) -> pd.Series:
    return series.pct_change() * 100.0

def pct_change_yoy(df: pd.DataFrame, freq: str) -> pd.Series:
    if freq == "M":
        return df["value"].pct_change(12) * 100.0
    else:
        return df["value"].pct_change(252) * 100.0

def normalize_100(series: pd.Series) -> pd.Series:
    idx = series.first_valid_index()
    if idx is None:
        return series
    base = series.loc[idx]
    return (series / base) * 100.0

def apply_transform(df: pd.DataFrame, freq: str, how: str) -> pd.DataFrame:
    out = df.copy()
    if how == "pct_mom":
        out["value"] = pct_change(out["value"])
    elif how == "pct_yoy":
        out["value"] = pct_change_yoy(out, freq)
    elif how == "normalize_100":
        out["value"] = normalize_100(out["value"])
    return out

def maybe_aggregate(df: pd.DataFrame, agg: str) -> pd.DataFrame:
    if agg == "native":
        return df
    x = df.set_index("date").sort_index()
    if agg == "M_last":
        x = x.resample("M").last()
    elif agg == "M_mean":
        x = x.resample("M").mean()
    x = x.reset_index()
    return x

def build_app(data: Dict[str, pd.DataFrame]) -> Dash:
    options = []
    for code in sorted(data.keys()):
        meta = SERIES.get(code, {"name": code})
        options.append({"label": f"{meta['name']} [{code}]", "value": code})

    first_code = next(iter(data.keys()))
    dmin = data[first_code]["date"].min()
    dmax = data[first_code]["date"].max()
    start_ui = pd.to_datetime(START_DATE).date() if START_DATE else dmin.date()
    end_ui = pd.to_datetime(END_DATE).date() if END_DATE else dmax.date()

    app = Dash(__name__)
    app.title = "US Macro (FRED) — Interactive"

    app.layout = html.Div([
        html.H2("US Macro (FRED) — интерактивные графики"),
        html.Div([
            html.Div([
                html.Label("Показатели (можно несколько)"),
                dcc.Dropdown(id="series-dd", options=options, value=["DGS10", "EFFR"], multi=True, style={"minWidth": "480px"}),
            ], style={"display": "inline-block", "marginRight": "16px", "verticalAlign": "top"}),

            html.Div([
                html.Label("Трансформация"),
                dcc.Dropdown(
                    id="transform-dd",
                    options=[{"label": v, "value": k} for k, v in TRANSFORMS.items()],
                    value="none",
                    clearable=False, style={"minWidth": "240px"}
                ),
            ], style={"display": "inline-block", "marginRight": "16px", "verticalAlign": "top"}),

            html.Div([
                html.Label("Агрегирование"),
                dcc.Dropdown(
                    id="agg-dd",
                    options=[{"label": v, "value": k} for k, v in AGG_OPTIONS.items()],
                    value="native",
                    clearable=False, style={"minWidth": "220px"}
                ),
            ], style={"display": "inline-block", "marginRight": "16px", "verticalAlign": "top"}),

            html.Div([
                html.Label("Диапазон дат"),
                dcc.DatePickerRange(
                    id="date-range",
                    start_date=str(start_ui),
                    end_date=str(end_ui),
                    display_format="YYYY-MM-DD"
                ),
            ], style={"display": "inline-block", "verticalAlign": "top"}),
        ], style={"marginBottom": "12px"}),

        dcc.Graph(id="macro-graph", style={"height": "72vh"}),

        html.Hr(),
        html.Div([
            html.Div("CSV сохраняются в ./data_fred:", style={"fontWeight": "bold"}),
            html.Ul([
                html.Li("индивидуальные файлы: <CODE>.csv"),
                html.Li("сводная длинная таблица: all_long.csv (с колонкой source)"),
                html.Li("сводная широкая таблица: all_wide.csv"),
            ])
        ], style={"fontFamily": "monospace"})
    ], style={"padding": "16px"})

    @app.callback(
        Output("macro-graph", "figure"),
        Input("series-dd", "value"),
        Input("transform-dd", "value"),
        Input("agg-dd", "value"),
        Input("date-range", "start_date"),
        Input("date-range", "end_date"),
    )
    def update_graph(codes: List[str], transform: str, agg: str, start_date: Optional[str], end_date: Optional[str]):
        if not codes:
            return go.Figure()

        traces = []
        for code in codes:
            if code not in data:
                continue
            meta = SERIES.get(code, {"name": code, "unit": "", "freq": "D"})
            df = data[code].copy()

            # фильтр по датам
            if start_date:
                df = df[df["date"] >= pd.to_datetime(start_date)]
            if end_date:
                df = df[df["date"] <= pd.to_datetime(end_date)]
            if df.empty:
                continue

            # агрегирование (до трансформации)
            df = maybe_aggregate(df, agg)
            # применить трансформацию
            freq_arg = ("M" if agg.startswith("M_") else meta["freq"])
            df_t = apply_transform(df, freq_arg, transform)

            traces.append(go.Scatter(
                x=df_t["date"],
                y=df_t["value"],
                mode="lines",
                name=f"{meta['name']} [{code}]",
                hovertemplate="%{x|%Y-%m-%d}<br>%{y:.6f}<extra>"+f"{code}</extra>"
            ))

        fig = go.Figure(traces)
        fig.update_layout(
            margin=dict(l=40, r=40, t=50, b=40),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
            hovermode="x unified",
            xaxis=dict(title=None, showgrid=True),
            yaxis=dict(title=f"{TRANSFORMS.get(transform,'none')} | {AGG_OPTIONS.get(agg,'native')}", showgrid=True),
        )
        return fig

    return app

def main():
    data = load_and_save_all()
    app = build_app(data)
    app.run(debug=True)

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        sys.exit(0)


=== Loading series ===
[OK] EFFR (FRED_API) → data_fred\EFFR.csv (rows=4478)
[OK] CPIAUCSL (FRED_API) → data_fred\CPIAUCSL.csv (rows=213)
[OK] UNRATE (FRED_API) → data_fred\UNRATE.csv (rows=212)
[OK] M2SL (FRED_API) → data_fred\M2SL.csv (rows=213)
[OK] DGS2 (FRED_API) → data_fred\DGS2.csv (rows=4458)
[OK] DGS10 (FRED_API) → data_fred\DGS10.csv (rows=4458)
[OK] DGS30 (FRED_API) → data_fred\DGS30.csv (rows=4458)
[OK] DCOILBRENTEU (FRED_API) → data_fred\DCOILBRENTEU.csv (rows=4503)
[WARN] fredapi failed for GOLDAMGBD228NLBM: Bad Request.  The series does not exist.. Trying FRED CSV …
[WARN] FRED CSV failed for GOLDAMGBD228NLBM: 404 Client Error: Not Found for url: https://fred.stlouisfed.org/graph/fredgraph.csv?id=GOLDAMGBD228NLBM&cosd=2008-01-01
[WARN] fredapi failed for GOLDPMGBD228NLBM: Bad Request.  The series does not exist.. Trying FRED CSV …


Failed to get ticker 'XAUUSD=X' reason: Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: C:\Users\allll\PycharmProjects\проект мага\.venv\Lib\site-packages\certifi\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

1 Failed download:
['XAUUSD=X']: SSLError('Failed to perform, curl: (77) error setting certificate verify locations:  CAfile: C:\\Users\\allll\\PycharmProjects\\проект мага\\.venv\\Lib\\site-packages\\certifi\\cacert.pem CApath: none. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


[WARN] FRED CSV failed for GOLDPMGBD228NLBM: 404 Client Error: Not Found for url: https://fred.stlouisfed.org/graph/fredgraph.csv?id=GOLDPMGBD228NLBM&cosd=2008-01-01
[WARN] FRED gold not available — falling back to Yahoo XAUUSD=X …
[ERROR] skipped GOLDAMGBD228NLBM: Yahoo returned empty XAUUSD=X
[OK] DEXUSEU (FRED_API) → data_fred\DEXUSEU.csv (rows=4460)
[WARN] Skipped series: GOLDAMGBD228NLBM
[DONE] Saved CSVs to: C:\Users\allll\PycharmProjects\проект мага\data_fred


2025-10-21    3.98
2025-10-22    3.97
2025-10-23    4.01
2025-10-24    4.02
2025-10-27    4.01
dtype: float64
